In [1]:
!pip -q install fastapi uvicorn pyngrok nest_asyncio "transformers>=4.43" "accelerate>=0.33" "bitsandbytes>=0.43"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

base_id = "mistralai/Mistral-Nemo-Instruct-2407"   
tok = AutoTokenizer.from_pretrained(base_id, use_fast=True)
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb_cfg, device_map="auto")
model.eval()

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[transformers] The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(131072, 5120)
    (layers): ModuleList(
      (0-39): 40 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=5120, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=5120, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=5120, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=5120, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=5120, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=5120, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((5120,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((5120,), eps=1e-05)
      )
    )
    (

In [3]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from typing import Optional, List, Dict

API_TOKEN = "secret123"       
app = FastAPI()

class Req(BaseModel):
    prompt: Optional[str] = None
    messages: Optional[List[Dict[str, str]]] = None
    max_length: int = 300
    temperature: float = 0.3

@app.post("/generate")
def generate(req: Req, authorization: str = Header(None)):
    if authorization != f"Bearer {API_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    msgs = req.messages or [{"role": "user", "content": req.prompt}]
    inputs = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]
    kw = dict(do_sample=True, temperature=req.temperature) if req.temperature > 0 else dict(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=req.max_length,
                             pad_token_id=tok.eos_token_id, **kw)
    return {"response": tok.decode(out[0][n_in:], skip_special_tokens=True)}

## 3) Expose with ngrok and serve (keep this cell running)

In [ ]:
import uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

ngrok.kill()   
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
public_url = ngrok.connect(8000).public_url
print("PASTE THIS IN STREAMLIT:", public_url)

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info"))
await server.serve()